<a href="https://colab.research.google.com/github/rymadinari/-arene-des-algos-Ryma-Dinari-/blob/main/Jour_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PHASE 1 : Split train / validation / test

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    val_size_ajuste = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_size_ajuste,
        random_state=random_state,
        stratify=y_temp
    )

    print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")
    print(f"Total vérifié : {len(X_train)+len(X_val)+len(X_test)} == {len(X)}")
    print(f"Répartition classe 1 :")
    print(f"  Train      : {y_train.mean():.2%}")
    print(f"  Validation : {y_val.mean():.2%}")
    print(f"  Test       : {y_test.mean():.2%}")
    print(f"  Original   : {y.mean():.2%}")

    return X_train, X_val, X_test, y_train, y_val, y_test


data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y)

Train : 341 | Validation : 114 | Test : 114
Total vérifié : 569 == 569
Répartition classe 1 :
  Train      : 62.76%
  Validation : 62.28%
  Test       : 63.16%
  Original   : 62.74%


In [4]:
# Checkpoints Phase 1

print("CHECKPOINT 1 — Tailles cohérentes")
total = len(X_train) + len(X_val) + len(X_test)
assert total == len(X), f"{total} != {len(X)}"
print(f"{len(X_train)} + {len(X_val)} + {len(X_test)} = {total}")

print()
# Checkpoint 2 : val_size=0 doit planter proprement
print("CHECKPOINT 2 — val_size=0")
try:
    split_train_val_test(X, y, val_size=0)
    print("Pas d'erreur — à gérer explicitement")
except Exception as e:
    print(f"Plante proprement : {e}")

print()
# Checkpoint 3 : dataset déséquilibré 95/5
print("CHECKPOINT 3 — Dataset déséquilibré 95/5")
y_dezequilibre = np.array([0]*950 + [1]*50)
X_dezequilibre = np.random.rand(1000, 5)
tr, val, te, ytr, yval, yte = split_train_val_test(
    X_dezequilibre, y_dezequilibre
)
print(f"Classe rare dans train : {ytr.mean():.2%}")
print(f"Classe rare dans val   : {yval.mean():.2%}")
print(f"Classe rare dans test  : {yte.mean():.2%}")
print("stratify conserve bien les 5% dans chaque jeu")

CHECKPOINT 1 — Tailles cohérentes
341 + 114 + 114 = 569

CHECKPOINT 2 — val_size=0
Plante proprement : The 'test_size' parameter of train_test_split must be a float in the range (0.0, 1.0), an int in the range [1, inf) or None. Got 0.0 instead.

CHECKPOINT 3 — Dataset déséquilibré 95/5
Train : 600 | Validation : 200 | Test : 200
Total vérifié : 1000 == 1000
Répartition classe 1 :
  Train      : 5.00%
  Validation : 5.00%
  Test       : 5.00%
  Original   : 5.00%
Classe rare dans train : 5.00%
Classe rare dans val   : 5.00%
Classe rare dans test  : 5.00%
stratify conserve bien les 5% dans chaque jeu


In [8]:
# PHASE 2 : Bootstrap et Bagging

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42):

    rng = np.random.default_rng(random_state)
    scores = []
    n = len(X)

    for i in range(n_iterations):

        indices_boot = rng.choice(n, size=n, replace=True)

        indices_oob = np.setdiff1d(np.arange(n), np.unique(indices_boot))

        if len(indices_oob) == 0:
            print(f"Itération {i+1} ignorée : aucun point OOB")
            continue

        X_boot = X[indices_boot]
        y_boot = y[indices_boot]

        X_oob = X[indices_oob]
        y_oob = y[indices_oob]

        modele.fit(X_boot, y_boot)

        y_pred = modele.predict(X_oob)
        score = accuracy_score(y_oob, y_pred)

        scores.append(score)

    scores = np.array(scores)

    if len(scores) == 1:
        print(f"Score bootstrap : {scores[0]:.3f}")
    else:
        print(
            f"Score moyen sur {len(scores)} bootstraps : "
            f"{scores.mean():.3f} (± {scores.std():.3f})"
        )

    return scores


scaler = StandardScaler()
X_s = scaler.fit_transform(X)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

scores_boot = bootstrap_scores(
    rf,
    X_s,
    y,
    n_iterations=30
)

print("\nTest avec une seule itération :")

bootstrap_scores(
    rf,
    X_s,
    y,
    n_iterations=1
)

print("\nComparaison replace=True vs replace=False :")

print(f"Avec remise : écart-type = {scores_boot.std():.4f}")

rng = np.random.default_rng(42)

idx = rng.choice(
    len(X),
    size=len(X),
    replace=False
)

oob = np.setdiff1d(np.arange(len(X)), idx)

if len(oob) == 0:
    print("Sans remise : aucun point OOB disponible.")
    print("Tous les exemples sont sélectionnés exactement une fois.")
    print("Ce n'est plus un bootstrap mais un simple mélange des données.")

Score moyen sur 30 bootstraps : 0.961 (± 0.015)

Test avec une seule itération :
Score bootstrap : 0.906

Comparaison replace=True vs replace=False :
Avec remise : écart-type = 0.0146
Sans remise : aucun point OOB disponible.
Tous les exemples sont sélectionnés exactement une fois.
Ce n'est plus un bootstrap mais un simple mélange des données.


In [9]:
# PHASE 3 : Validation croisée k-fold

from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

def evaluer_en_cross_val(modele, X, y, k=5):
    """Lance une validation croisée k-fold et résume les résultats."""
    scores = cross_val_score(modele, X, y, cv=k, scoring="accuracy")

    print(f"Scores par fold : {np.round(scores, 3)}")
    print(f"Moyenne : {scores.mean():.3f} | Écart-type : {scores.std():.3f}", end="  ->  ")

    if scores.std() < 0.02:
        print("modèle stable ")
    else:
        print("modèle instable ")

    return scores


print("CROSS-VAL 5-fold :")
evaluer_en_cross_val(rf, X_s, y, k=5)

CROSS-VAL 5-fold :
Scores par fold : [0.921 0.939 0.982 0.965 0.973]
Moyenne : 0.956 | Écart-type : 0.023  ->  modèle instable 


array([0.92105263, 0.93859649, 0.98245614, 0.96491228, 0.97345133])

In [13]:
#Checkpoints Phase 3

from sklearn.model_selection import cross_val_score, LeaveOneOut, StratifiedKFold

# Checkpoint 1 : 5 folds normal
print("CHECKPOINT 1 — 5 folds")
evaluer_en_cross_val(rf, X_s, y, k=5)


# Checkpoint 2 : leave-one-out (k = nb lignes)
print("\nCHECKPOINT 2 — Leave-One-Out (lent !)")
loo = LeaveOneOut()

start = time.time()

scores_loo = cross_val_score(
    rf,
    X_s,
    y,
    cv=loo,
    scoring="accuracy"
)

end = time.time()

print(f"Moyenne LOO : {scores_loo.mean():.3f}")
print(f"Temps d'exécution : {end - start:.2f} secondes")

print("Pourquoi c'est lent ?")
print("- On entraîne le modèle N fois (N = nombre d'exemples)")
print("- Chaque entraînement est presque identique")
print("- Coût très élevé pour peu de gain en stabilité")



# Checkpoint 3 : déséquilibré — standard vs stratified
print("\nCHECKPOINT 3 — KFold standard vs StratifiedKFold sur 95/5")
skf = StratifiedKFold(n_splits=5)
scores_std  = cross_val_score(rf, X_dezequilibre, y_dezequilibre, cv=5)
scores_strat = cross_val_score(rf, X_dezequilibre, y_dezequilibre, cv=skf)
print(f"Standard   : {scores_std.mean():.3f} ± {scores_std.std():.3f}")
print(f"Stratifié  : {scores_strat.mean():.3f} ± {scores_strat.std():.3f}")
print("Sans stratification, certains folds peuvent n'avoir aucun cas rare")

CHECKPOINT 1 — 5 folds
Scores par fold : [0.921 0.939 0.982 0.965 0.973]
Moyenne : 0.956 | Écart-type : 0.023  ->  modèle instable 

CHECKPOINT 2 — Leave-One-Out (lent !)
Moyenne LOO : 0.961
Temps d'exécution : 198.54 secondes
Pourquoi c'est lent ?
- On entraîne le modèle N fois (N = nombre d'exemples)
- Chaque entraînement est presque identique
- Coût très élevé pour peu de gain en stabilité

CHECKPOINT 3 — KFold standard vs StratifiedKFold sur 95/5
Standard   : 0.950 ± 0.000
Stratifié  : 0.950 ± 0.000
Sans stratification, certains folds peuvent n'avoir aucun cas rare
